# Review Generated Schema.org JSON-LD

Inspect the Gemini 2.5 Flash generation output before moving to training data prep.

Checks:
1. Overall valid/invalid/failed breakdown
2. Quality score distribution
3. Type distribution (generated vs hint)
4. Property count stats
5. Random samples per type for manual review
6. Common failure modes

In [ ]:
import json, glob, random, os
from pathlib import Path
from collections import Counter
from IPython.display import display, HTML, JSON

GEN_DIR = Path('../data/generated')
files = sorted(glob.glob(str(GEN_DIR / '*.json')))
print(f'Total generated files: {len(files):,}')

In [ ]:
# Load all results
results = []
for f in files:
    with open(f) as fh:
        results.append(json.load(fh))

valid   = [r for r in results if r['valid']]
invalid = [r for r in results if not r['valid']]

print(f'Valid:   {len(valid):,} ({len(valid)/len(results)*100:.1f}%)')
print(f'Invalid: {len(invalid):,} ({len(invalid)/len(results)*100:.1f}%)')

## Quality Score Distribution

In [ ]:
scores = [r['quality_score'] for r in valid]

# Histogram buckets
buckets = Counter()
for s in scores:
    bucket = round(s, 1)
    buckets[bucket] += 1

print(f'Quality score stats:')
print(f'  Min:    {min(scores):.3f}')
print(f'  Max:    {max(scores):.3f}')
print(f'  Mean:   {sum(scores)/len(scores):.3f}')
print(f'  Median: {sorted(scores)[len(scores)//2]:.3f}')
print()
print('Distribution:')
for bucket in sorted(buckets.keys()):
    bar = '#' * (buckets[bucket] // 20)
    print(f'  {bucket:.1f}  {buckets[bucket]:5,}  {bar}')

## Type Distribution

In [ ]:
# What types did Gemini actually generate vs what we hinted?
hint_types = Counter(r['schema_type_hint'] for r in valid)
gen_types  = Counter(r['schema_types'][0] for r in valid if r['schema_types'])

all_types = sorted(set(hint_types.keys()) | set(gen_types.keys()), 
                   key=lambda t: -gen_types.get(t, 0))

print(f'{"Generated Type":35s} {"Count":>6s}   {"Hint Type":35s} {"Count":>6s}')
print('-' * 90)
for i, t in enumerate(all_types[:25]):
    gen_str  = f'{t:35s} {gen_types.get(t, 0):6,}'
    hint_str = ''
    if i < len(hint_types.most_common(25)):
        ht, hc = hint_types.most_common(25)[i]
        hint_str = f'{ht:35s} {hc:6,}'
    print(f'{gen_str}   {hint_str}')

print(f'\nTotal unique generated types: {len(gen_types)}')
print(f'Total unique hint types:     {len(hint_types)}')

## Property Count Stats

In [ ]:
props = [r['property_count'] for r in valid]

print(f'Property count stats (valid only):')
print(f'  Min:    {min(props)}')
print(f'  Max:    {max(props)}')
print(f'  Mean:   {sum(props)/len(props):.1f}')
print(f'  Median: {sorted(props)[len(props)//2]}')
print()

# By type
props_by_type = {}
for r in valid:
    t = r['schema_types'][0] if r['schema_types'] else '?'
    props_by_type.setdefault(t, []).append(r['property_count'])

print(f'{"Type":30s} {"Count":>6s} {"Avg Props":>10s} {"Min":>5s} {"Max":>5s}')
print('-' * 60)
for t in sorted(props_by_type, key=lambda x: -len(props_by_type[x]))[:20]:
    p = props_by_type[t]
    print(f'{t:30s} {len(p):6,} {sum(p)/len(p):10.1f} {min(p):5d} {max(p):5d}')

## Sample Review (3 per major type)

Manually check that generated schema looks correct.

In [ ]:
random.seed(123)

# Group valid results by generated type
by_type = {}
for r in valid:
    t = r['schema_types'][0] if r['schema_types'] else '?'
    by_type.setdefault(t, []).append(r)

# Show top types
review_types = [t for t, _ in Counter({t: len(v) for t, v in by_type.items()}).most_common(10)]

for schema_type in review_types:
    pool = by_type[schema_type]
    sample = random.sample(pool, min(3, len(pool)))
    
    print(f'\n{"=" * 70}')
    print(f'{schema_type} ({len(pool):,} total)')
    print(f'{"=" * 70}')
    
    for r in sample:
        print(f'\nURL: {r["url"][:80]}')
        print(f'Quality: {r["quality_score"]}  |  Props: {r["property_count"]}')
        print('-' * 50)
        # Pretty print the JSON-LD
        try:
            parsed = json.loads(r['generated_jsonld'])
            print(json.dumps(parsed, indent=2, ensure_ascii=False)[:1500])
            if len(r['generated_jsonld']) > 1500:
                print(f'... [{len(r["generated_jsonld"]) - 1500} more chars]')
        except:
            print(r['generated_jsonld'][:1500])
        print()

## Failure Analysis

In [ ]:
error_types = Counter()
for r in invalid:
    for e in r.get('errors', []):
        # Normalize error messages
        if 'Unterminated string' in e:
            error_types['Truncated output (unterminated string)'] += 1
        elif 'Missing required' in e:
            error_types[e] += 1
        elif 'Invalid JSON' in e:
            error_types['Invalid JSON (other)'] += 1
        else:
            error_types[e[:60]] += 1

print(f'Error breakdown ({len(invalid)} invalid results):')
print()
for err, cnt in error_types.most_common():
    print(f'  {cnt:5,}  {err}')

In [ ]:
# Show a few invalid examples to understand patterns
print('Sample invalid outputs:\n')
for r in random.sample(invalid, min(5, len(invalid))):
    print(f'URL: {r["url"][:70]}')
    print(f'Hint: {r["schema_type_hint"]}  |  Errors: {r["errors"]}')
    print(f'Last 150 chars: ...{r["generated_jsonld"][-150:]}')
    print()

## Summary

In [ ]:
high_q = [r for r in valid if r['quality_score'] >= 0.4]

print('=' * 50)
print('GENERATION SUMMARY')
print('=' * 50)
print(f'Total generated:  {len(results):,}')
print(f'Valid:            {len(valid):,} ({len(valid)/len(results)*100:.1f}%)')
print(f'High quality:     {len(high_q):,} ({len(high_q)/len(results)*100:.1f}%)  (score >= 0.4)')
print(f'Invalid:          {len(invalid):,}')
print(f'Unique gen types: {len(gen_types)}')
print(f'Avg properties:   {sum(props)/len(props):.1f}')
print()
print('Ready for training data prep (notebook 05)?')
if len(high_q) >= 5000:
    print(f'  YES - {len(high_q):,} high-quality examples is sufficient.')
else:
    print(f'  MAYBE - {len(high_q):,} examples may be low. Consider re-running failed pages.')